In [1]:
# ** This cell is needed since we are not in the src directory **
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': True,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8]},
 'method': 'welch',
 'stepSize': 1.5,
 'windowLength': 3}

In [3]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [5]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [6]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx16g -Xms8g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # SPARK DIRECTORY FOR THREADS / PERSIST
    
    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \
    
    # THIS IS WHERE SPARK WILL PUT ITS TEMPERARY VARIABLES 
    # .config("spark.local.dir", os.path.expanduser("~/external-spark-tmp"))

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "512") \

    # prevent breaking pipes 
    .config("spark.reducer.maxReqsInFlight", "1") \
    .config("spark.shuffle.io.preferDirectBufs", "false") \
    .config("spark.shuffle.file.buffer", "32k") \
    
    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx16g -Xms8g
Picked up _JAVA_OPTIONS: -Xmx16g -Xms8g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/04 14:24:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


New Spark session created successfully


In [7]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Error adding files to SparkContext: An error occurred while calling o46.addFile.
: java.io.FileNotFoundException: File file:/Volumes/CrucialX6/Home/projects/eeg-ds004504/..src/feature_extraction.py does not exist
	at org.apach

In [8]:
 parquet_direcotry = "/Volumes/CrucialX6/Home/Desktop/eeg-ds004504-BACKUP/notebooks"

In [9]:
alz_df_spark = spark.read.parquet(f"{parquet_direcotry}/features_alz_extra_features_Apr19_2141.parquet")
cntrl_df_spark = spark.read.parquet(f"{parquet_direcotry}/features_cntrl_extra_features_Apr19_2141.parquet")

In [10]:
# rename for useability, I wanted it to be clear that the dataframes are spark.sql types when we create them above
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

In [11]:
alz_df.show()

+---------+-------+---------+--------+-----------+-------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName| FeatureValue|table_type|
+---------+-------+---------+--------+-----------+-------------+----------+
|  sub-008|   ep-0|      Fp1|   Alpha|      Power|  7.020328E-4|      band|
|  sub-008|   ep-0|      Fp1|    Beta|      Power|  3.399499E-4|      band|
|  sub-008|   ep-0|      Fp1|   Delta|      Power|  0.087231696|      band|
|  sub-008|   ep-0|      Fp1|   Theta|      Power| 0.0011391409|      band|
|  sub-008|   ep-0|      Fp1| custom1|      Power| 4.2795436E-4|      band|
|  sub-008|   ep-0|      Fp1|    NULL|TotalEnergy|   0.34805238| electrode|
|  sub-008|   ep-0|      Fp1|    NULL| TotalPower|  0.011235955| electrode|
|  sub-008|   ep-0|      Fp2|   Alpha|      Power| 0.0014689578|      band|
|  sub-008|   ep-0|      Fp2|    Beta|      Power|  5.043481E-4|      band|
|  sub-008|   ep-0|      Fp2|   Delta|      Power|   0.08462298|      band|
|  sub-008| 

# Data Processing and Dimensionality Reduction

In [12]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [13]:
full_df = alz_df.unionByName(cntrl_df)

In [14]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")

In [15]:
from pyspark.sql.functions import concat_ws

# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName")).repartition(16).persist()

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName")).repartition(16).persist()

# Epoch-level: just FeatureName
epoch_df = epoch_df.withColumn("pivot", col("FeatureName")).repartition(16).persist()


In [16]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

In [17]:
from functools import reduce



full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()

25/10/04 14:24:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: float, C3_Beta_Power: float, C3_Delta_Power: float, C3_Theta_Power: float, C3_custom1_Power: float, C4_Alpha_Power: float, C4_Beta_Power: float, C4_Delta_Power: float, C4_Theta_Power: float, C4_custom1_Power: float, Cz_Alpha_Power: float, Cz_Beta_Power: float, Cz_Delta_Power: float, Cz_Theta_Power: float, Cz_custom1_Power: float, F3_Alpha_Power: float, F3_Beta_Power: float, F3_Delta_Power: float, F3_Theta_Power: float, F3_custom1_Power: float, F4_Alpha_Power: float, F4_Beta_Power: float, F4_Delta_Power: float, F4_Theta_Power: float, F4_custom1_Power: float, F7_Alpha_Power: float, F7_Beta_Power: float, F7_Delta_Power: float, F7_Theta_Power: float, F7_custom1_Power: float, F8_Alpha_Power: float, F8_Beta_Power: float, F8_Delta_Power: float, F8_Theta_Power: float, F8_custom1_Power: float, Fp1_Alpha_Power: float, Fp1_Beta_Power: float, Fp1_Delta_Power: float, Fp1_Theta_Power: float, Fp1_custom1_Power: float, Fp2_Alpha

In [18]:
full_df.repartition(16).persist()

25/10/04 14:24:53 WARN CacheManager: Asked to cache already cached data.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: float, C3_Beta_Power: float, C3_Delta_Power: float, C3_Theta_Power: float, C3_custom1_Power: float, C4_Alpha_Power: float, C4_Beta_Power: float, C4_Delta_Power: float, C4_Theta_Power: float, C4_custom1_Power: float, Cz_Alpha_Power: float, Cz_Beta_Power: float, Cz_Delta_Power: float, Cz_Theta_Power: float, Cz_custom1_Power: float, F3_Alpha_Power: float, F3_Beta_Power: float, F3_Delta_Power: float, F3_Theta_Power: float, F3_custom1_Power: float, F4_Alpha_Power: float, F4_Beta_Power: float, F4_Delta_Power: float, F4_Theta_Power: float, F4_custom1_Power: float, F7_Alpha_Power: float, F7_Beta_Power: float, F7_Delta_Power: float, F7_Theta_Power: float, F7_custom1_Power: float, F8_Alpha_Power: float, F8_Beta_Power: float, F8_Delta_Power: float, F8_Theta_Power: float, F8_custom1_Power: float, Fp1_Alpha_Power: float, Fp1_Beta_Power: float, Fp1_Delta_Power: float, Fp1_Theta_Power: float, Fp1_custom1_Power: float, Fp2_Alpha

In [19]:
fraction = 0.2  # fraction to KEEP
seed = 42       # random seed for reproducibility (any integer)
full_df = full_df.sample(withReplacement=False, fraction=fraction, seed=seed)

In [20]:
# full_df.createOrReplaceTempView("full_df")   # now SQL can SELECT FROM epochs

In [21]:
# train_df = spark.sql("""
# WITH ranked AS (
#   SELECT *,
#          ntile(5) OVER (PARTITION BY SubjectID ORDER BY rand(42)) AS bucket
#   FROM full_df
# )
# SELECT * FROM ranked WHERE bucket <= 4
# """)

# test_df = spark.sql("""
# WITH ranked AS (
#   SELECT *,
#          ntile(5) OVER (PARTITION BY SubjectID ORDER BY rand(42)) AS bucket
#   FROM full_df
# )
# SELECT * FROM ranked WHERE bucket = 5
# """)

In [22]:
# print("all:", full_df.count(),
#       "\ntrain:", train_df.count(),
#       "\ntest:", test_df.count())

# overlap = (train_df.select("SubjectID","EpochID").intersect(
#            test_df.select("SubjectID","EpochID"))).count()
# print("overlap rows:", overlap)

# START TRANSFORMATIONS (WILL NEED TO CREATE DENSEVECTOR FOR FEATURE COLUMNS

In [23]:
full_df.show()

[Stage 113:>                                                      (0 + 12) / 12]

+---------+-------+-----+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+--------------+----------------+---------------+--------------+---------------+---------------+-----------------+---------------+--------------+---------------+---------------+-----------------+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+--------------+----------------+--------------+-------------+--------------+---------

In [24]:
from pyspark.ml.feature import VectorAssembler
feature_cols = [c for c in full_df.columns if c not in ("SubjectID", "EpochID", "label")]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
vector_df = assembler.transform(full_df)
full_df = vector_df.select("SubjectID", "EpochID", "label", "features")



In [25]:
full_df.show()

[Stage 150:==================================================>    (11 + 1) / 12]

+---------+-------+-----+--------------------+
|SubjectID|EpochID|label|            features|
+---------+-------+-----+--------------------+
|  sub-001|ep-1075|    1|[8.00567038822919...|
|  sub-001| ep-119|    1|[0.00124174018856...|
|  sub-001|ep-1210|    1|[0.00120238936506...|
|  sub-001|  ep-13|    1|[0.00119471130892...|
|  sub-001|ep-1427|    1|[0.00253925938159...|
|  sub-001|ep-1447|    1|[6.45557476673275...|
|  sub-001|ep-1523|    1|[0.00246033049188...|
|  sub-001|ep-1558|    1|[0.00132785364985...|
|  sub-001|ep-1670|    1|[0.00168819387909...|
|  sub-001|ep-1758|    1|[4.90785227157175...|
|  sub-001|ep-1790|    1|[0.00256728217937...|
|  sub-001| ep-180|    1|[0.00125925114843...|
|  sub-001| ep-181|    1|[0.00165384181309...|
|  sub-001|ep-1854|    1|[0.00302652362734...|
|  sub-001|ep-1944|    1|[5.44302281923592...|
|  sub-001| ep-231|    1|[0.00502884481102...|
|  sub-001| ep-256|    1|[9.30653070099651...|
|  sub-001| ep-351|    1|[0.00162692391313...|
|  sub-001| e

In [26]:
# first ... min max 
from pyspark.ml.feature import VectorAssembler, MinMaxScaler, PCA
# MinMaxScaler expects the input column to be a vector ("features")
scaler1 = MinMaxScaler(inputCol="features", outputCol="scaled_features")
scaled_df = scaler1.fit(full_df).transform(full_df)

# second ... pca 
pca = PCA(k=10, inputCol="scaled_features", outputCol="pca_features")
pca_model = pca.fit(scaled_df)
pca_df = pca_model.transform(scaled_df)

# third ... min max 
scaler2 = MinMaxScaler(inputCol="pca_features", outputCol="final_features")
final_scaled_df = scaler2.fit(pca_df).transform(pca_df)

# select relavent columns only 
full_df = final_scaled_df.select("SubjectID", "EpochID", "label", "final_features") \
    .withColumnRenamed("final_features", "features")

25/10/04 14:25:28 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/10/04 14:25:28 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
                                                                                

In [27]:
# split test/train


full_df.createOrReplaceTempView("full_df")   # now SQL can SELECT FROM epochs
train_df = spark.sql("""
WITH ranked AS (
  SELECT *,
         ntile(5) OVER (PARTITION BY SubjectID ORDER BY rand(42)) AS bucket
  FROM full_df
)
SELECT * FROM ranked WHERE bucket <= 4
""")

test_df = spark.sql("""
WITH ranked AS (
  SELECT *,
         ntile(5) OVER (PARTITION BY SubjectID ORDER BY rand(42)) AS bucket
  FROM full_df
)
SELECT * FROM ranked WHERE bucket = 5
""")

print("all:", full_df.count(),
      "\ntrain:", train_df.count(),
      "\ntest:", test_df.count())

overlap = (train_df.select("SubjectID","EpochID").intersect(
           test_df.select("SubjectID","EpochID"))).count()
print("overlap rows:", overlap)

all: 35887 
train: 28736 
test: 7151


[Stage 507:=============================================>         (10 + 2) / 12]

overlap rows: 0


In [28]:
train_pd_df = train_df.toPandas()
test_pd_df = test_df.toPandas()

In [29]:
train_pd_df

,SubjectID,EpochID,label,features,bucket
0,sub-008,ep-793,1,"[0.7565336044114525, 0.787068743488864, 0.4313...",1
1,sub-008,ep-737,1,"[0.6849852911505213, 0.8592468344926623, 0.433...",1
2,sub-008,ep-2571,1,"[0.7650058289495454, 0.6845750702206531, 0.391...",1
3,sub-008,ep-1269,1,"[0.8069053903105274, 0.6584970005002233, 0.395...",1
4,sub-008,ep-570,1,"[0.5976911576652699, 0.79678506215907, 0.36574...",1
...,...,...,...,...,...
28731,sub-059,ep-2408,0,"[0.5766274501692898, 0.7904256335295217, 0.368...",4
28732,sub-059,ep-1146,0,"[0.6536752380485272, 0.885721729019401, 0.4491...",4
28733,sub-059,ep-298,0,"[0.7804385429249999, 0.6723542792174532, 0.390...",4
28734,sub-059,ep-2548,0,"[0.7762157797129148, 0.7492211493386559, 0.425...",4


# END TRANSFORMATIONS

In [30]:
# Machine Learning results are below 

In [31]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import recall_score
import numpy as np

def subject_knn_recall(train_df, test_df, k=1):
    # Ensure features are numpy arrays
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df['features'] = train_df['features'].apply(lambda x: np.array(x))
    test_df['features'] = test_df['features'].apply(lambda x: np.array(x))

    X_train = np.stack(train_df['features'].values)
    y_train = train_df['SubjectID'].values

    X_test = np.stack(test_df['features'].values)
    y_test = test_df['SubjectID'].values

    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    recall = recall_score(y_test, y_pred, average='macro')
    print(f"Recall (macro): {recall:.4f}")
    return recall


In [32]:
def progressive_subject_knn_recall(train_df, test_df, max_subjects=20, k=1, random_state=42):
    # Find subjects present in both sets
    common_subjects = np.intersect1d(train_df['SubjectID'].unique(), test_df['SubjectID'].unique())
    rng = np.random.default_rng(random_state)
    rng.shuffle(common_subjects)

    results = []

    for n_subjects in range(2, min(max_subjects+1, len(common_subjects)+1)):
        sel_subjects = common_subjects[:n_subjects]

        tr = train_df[train_df['SubjectID'].isin(sel_subjects)]
        te = test_df[test_df['SubjectID'].isin(sel_subjects)]

        X_tr = np.stack(tr['features'].values)
        y_tr = tr['SubjectID'].values
        X_te = np.stack(te['features'].values)
        y_te = te['SubjectID'].values

        clf = KNeighborsClassifier(n_neighbors=k)
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)
        recall = recall_score(y_te, y_pred, average='macro')

        results.append((n_subjects, recall))
        print(f"{n_subjects} subjects — Recall (macro): {recall:.4f}")

    return pd.DataFrame(results, columns=["Num_Subjects", "Recall"])
recall_df = progressive_subject_knn_recall(train_pd_df, test_pd_df, max_subjects=65, k=1)

2 subjects — Recall (macro): 0.9216
3 subjects — Recall (macro): 0.8069
4 subjects — Recall (macro): 0.8106
5 subjects — Recall (macro): 0.7552
6 subjects — Recall (macro): 0.6891
7 subjects — Recall (macro): 0.6553
8 subjects — Recall (macro): 0.6522
9 subjects — Recall (macro): 0.6130
10 subjects — Recall (macro): 0.5473
11 subjects — Recall (macro): 0.5462
12 subjects — Recall (macro): 0.5552
13 subjects — Recall (macro): 0.5894
14 subjects — Recall (macro): 0.5748
15 subjects — Recall (macro): 0.5511
16 subjects — Recall (macro): 0.5233
17 subjects — Recall (macro): 0.5257
18 subjects — Recall (macro): 0.5006
19 subjects — Recall (macro): 0.4822
20 subjects — Recall (macro): 0.4715
21 subjects — Recall (macro): 0.4567
22 subjects — Recall (macro): 0.4403
23 subjects — Recall (macro): 0.4308
24 subjects — Recall (macro): 0.4519
25 subjects — Recall (macro): 0.4409
26 subjects — Recall (macro): 0.4311
27 subjects — Recall (macro): 0.4236
28 subjects — Recall (macro): 0.4066
29 subjec

In [33]:
recall_df = progressive_subject_knn_recall(train_pd_df, test_pd_df, max_subjects=65, k=3)

2 subjects — Recall (macro): 0.9191
3 subjects — Recall (macro): 0.7946
4 subjects — Recall (macro): 0.7968
5 subjects — Recall (macro): 0.7275
6 subjects — Recall (macro): 0.6531
7 subjects — Recall (macro): 0.6333
8 subjects — Recall (macro): 0.6235
9 subjects — Recall (macro): 0.5823
10 subjects — Recall (macro): 0.5328
11 subjects — Recall (macro): 0.5331
12 subjects — Recall (macro): 0.5389
13 subjects — Recall (macro): 0.5744
14 subjects — Recall (macro): 0.5608
15 subjects — Recall (macro): 0.5377
16 subjects — Recall (macro): 0.5075
17 subjects — Recall (macro): 0.5062
18 subjects — Recall (macro): 0.4748
19 subjects — Recall (macro): 0.4519
20 subjects — Recall (macro): 0.4458
21 subjects — Recall (macro): 0.4235
22 subjects — Recall (macro): 0.4059
23 subjects — Recall (macro): 0.3961
24 subjects — Recall (macro): 0.4198
25 subjects — Recall (macro): 0.4043
26 subjects — Recall (macro): 0.3979
27 subjects — Recall (macro): 0.3900
28 subjects — Recall (macro): 0.3808
29 subjec

In [34]:
recall_df = progressive_subject_knn_recall(train_pd_df, test_pd_df, max_subjects=65, k=5)

2 subjects — Recall (macro): 0.9239
3 subjects — Recall (macro): 0.8116
4 subjects — Recall (macro): 0.8224
5 subjects — Recall (macro): 0.7434
6 subjects — Recall (macro): 0.6828
7 subjects — Recall (macro): 0.6492
8 subjects — Recall (macro): 0.6448
9 subjects — Recall (macro): 0.6027
10 subjects — Recall (macro): 0.5554
11 subjects — Recall (macro): 0.5469
12 subjects — Recall (macro): 0.5597
13 subjects — Recall (macro): 0.5936
14 subjects — Recall (macro): 0.5760
15 subjects — Recall (macro): 0.5577
16 subjects — Recall (macro): 0.5240
17 subjects — Recall (macro): 0.5244
18 subjects — Recall (macro): 0.4940
19 subjects — Recall (macro): 0.4788
20 subjects — Recall (macro): 0.4701
21 subjects — Recall (macro): 0.4554
22 subjects — Recall (macro): 0.4377
23 subjects — Recall (macro): 0.4310
24 subjects — Recall (macro): 0.4517
25 subjects — Recall (macro): 0.4385
26 subjects — Recall (macro): 0.4281
27 subjects — Recall (macro): 0.4161
28 subjects — Recall (macro): 0.4009
29 subjec

In [ ]:
recall_df = progressive_subject_knn_recall(train_pd_df, test_pd_df, max_subjects=65, k=10)

In [ ]:
recall_df = progressive_subject_knn_recall(train_pd_df, test_pd_df, max_subjects=65, k=30)